# Trajectories of magnetic soft robot

### Imports

In [1]:
import numpy as np
import jax.numpy as jnp
import jax
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from softmobility import SoftBody, FlowBodySolver, shear_flow, oscillating_magnetic_field

np.set_printoptions(precision=3, linewidth=100, suppress=True, sign=" ")

In [109]:
MAGNETIC_FORCING = 100  # Magnetic amplitude: m Bx / (mu a omega)
MAGNETIC_RATIO = 1     # Magnetic ratio: By /Bx

## Numerical approach

### Parameter file

In [110]:
yaml_data = """
# Variable Prefixes (Optional)
dof_names:
  - phi
design_names:      
  - spring_k    
  - radius
  - length
input_names:
  - magnetic
    
# Default Values (Optional)
defaults:
  refradius: 1
  moment: 1
  spring_k: 84            
  radius: 1.
  length: 0.1

# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: refradius                    
    orientation: [0, 0, phi]
    torque: 
      - 0
      - 0
      - "moment * magnetic1 * cos(phi) - moment * magnetic0 * sin(phi) - spring_k * phi"              

  - # Sphere 1 #################
    radius: radius               
    position: [refradius + radius + length, 0, 0]       
    torque: [0, 0, spring_k * phi]    
"""

### Soft Body

In [111]:
mybody= SoftBody(yaml_data, verbose=True)

  Found variables: phi, length, radius, spring_k, magnetic0, magnetic1, magnetic2
  3D field inputs:  ['magnetic']
    Sphere 0
      Radius: 1
      Position: ['0', '0', '0']
      Orientation: ['0', '0', 'phi']
      Coupling matrix C_H:
[['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['-sin(phi)', 'cos(phi)', '0']]
      Coupling matrix C_K:
[['0'], ['0'], ['0'], ['0'], ['0'], ['-spring_k']]
    Sphere 1
      Radius: radius
      Position: ['length + radius + 1', '0', '0']
      Orientation: ['0', '0', '0']
      Coupling matrix C_H:
[['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0']]
      Coupling matrix C_K:
[['0'], ['0'], ['0'], ['0'], ['0'], ['spring_k']]


### Flow-body solver

In [112]:
# Create a shear flow with shear rate 1
no_flow = shear_flow(shear_rate=0)
# Create a magnetic field 
magnetic_field = oscillating_magnetic_field(amp_x=MAGNETIC_FORCING, amp_y=MAGNETIC_RATIO * MAGNETIC_FORCING, omega=1)

print(magnetic_field.vector(time=0))
print(magnetic_field.vector(time=0.2))

[ 100.    0.    0.]
[ 100.      19.867    0.   ]


In [ ]:
# parameters
final_time = 12 * 2 * jnp.pi  
dt = 0.1

# optimal design parameters
mybody.set_design_defaults(new_dict={'spring_k': 35., 'radius': .68, 'length': 0.})

solver = FlowBodySolver(
    soft_body=mybody, 
    flow=no_flow, 
    input_map={"magnetic": magnetic_field}, 
    dt=dt, 
    init_orientation=[0, 0, 0], 
    integrator="RK2")
trajectory = solver.simulate(final_time=final_time)

OLD default param values: ['length', 'radius', 'spring_k'] [  0.     0.68  35.  ]
NEW default param values: ['length', 'radius', 'spring_k'] [  0.    0.7  35. ]
Time: 0.000 / 75.398  Integrator RK2
Time: 10.000 / 75.398  Integrator RK2
Time: 20.000 / 75.398  Integrator RK2
Time: 30.000 / 75.398  Integrator RK2
Time: 40.000 / 75.398  Integrator RK2
Time: 50.000 / 75.398  Integrator RK2
Time: 60.000 / 75.398  Integrator RK2
Time: 70.000 / 75.398  Integrator RK2


### Figure of trajectory

In [130]:
# Extract position, orientation and dofs
positions = jnp.array([t[0] for t in trajectory])
orientations = jnp.array([t[1] for t in trajectory])
dofs = jnp.array([t[2][0] for t in trajectory])

# Time vector
t = jnp.linspace(0, final_time, len(trajectory))

# Create a subplot figure with 3 rows and 1 column
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=("DOF", "Position (X, Y, Z)", "Orientation (X, Y, Z)"),
    shared_xaxes=True,  # Share the x-axis for all plots (time)
    vertical_spacing=0.1  # Adjust vertical spacing (reduce clutter)
)

# Plot DOFs in the first subplot
fig.add_trace(go.Scatter(x=t, y=dofs[:], mode='lines', name='DOF 0'), row=1, col=1)

# Plot position components (X, Y, Z) in the second subplot
fig.add_trace(go.Scatter(x=t, y=positions[:, 0], mode='lines', name='Position X'), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=positions[:, 1], mode='lines', name='Position Y'), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=positions[:, 2], mode='lines', name='Position Z'), row=2, col=1)

# Plot orientation components (X, Y, Z) in the third subplot
fig.add_trace(go.Scatter(x=t, y=orientations[:, 0], mode='lines', name='Orientation X'), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=orientations[:, 1], mode='lines', name='Orientation Y'), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=orientations[:, 2], mode='lines', name='Orientation Z'), row=3, col=1)

# Update layout for titles and labels
fig.update_layout(
    title="Trajectory Data Over Time",
    xaxis_title="Time (t)",
    # yaxis_title="Values (Position/Orientation Components)",
    template="plotly_white",  # Set the background to white
    showlegend=True,  # Show legend to distinguish between the traces
    height=800,  # Set figure height
    width=800,   # Set figure width
    plot_bgcolor='white',  # Set the plot background to white
    paper_bgcolor='white'  # Set the paper background to white
)

# Show the plot
fig.show()

## Theoretical approach, limit of small angles

### Parameter file

In [101]:
yaml_data = """
# Variable Prefixes (Optional)
dof_names:
  - phi
design_names:      
  - stiffness    
  - radius
  - length
input_names:
  - magnetic
    
# Default Values (Optional)
defaults:
  stiffness: 0.1            
  radius: 0.25
  length: 0.5
  stiffness_factor: 100
  radius_factor: 2
  length_factor: 2
  moment: 1
  refradius: 1
  
# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: refradius                    
    orientation: [0, 0, phi]
    torque: 
      - 0
      - 0
      - "moment * magnetic - stiffness_factor * stiffness * phi"              

  - # Sphere 1 #################
    radius: radius_factor * radius               
    position: [refradius + radius_factor * radius + length_factor * length, 0, 0]       
    torque: [0, 0, stiffness_factor * stiffness * phi]    
"""

### Extracting mobility components

In [102]:
theory_body= SoftBody(yaml_data, verbose=True)

  Found variables: phi, length, radius, stiffness, magnetic
  Scalar inputs:    ['magnetic']
    Sphere 0
      Radius: 1
      Position: ['0', '0', '0']
      Orientation: ['0', '0', 'phi']
      Coupling matrix C_H:
[['0'], ['0'], ['0'], ['0'], ['0'], ['1']]
      Coupling matrix C_K:
[['0'], ['0'], ['0'], ['0'], ['0'], ['-100*stiffness']]
    Sphere 1
      Radius: 2*radius
      Position: ['2*length + 2*radius + 1', '0', '0']
      Orientation: ['0', '0', '0']
      Coupling matrix C_H:
[['0'], ['0'], ['0'], ['0'], ['0'], ['0']]
      Coupling matrix C_K:
[['0'], ['0'], ['0'], ['0'], ['0'], ['100*stiffness']]


In [103]:
tensors = theory_body.compute_mobility_problem()
print("\nMobility Matrix M_H (multiplied by mu):")
print(tensors.M_H)
print("\nMobility Matrix M_K (multiplied by mu):")
print(tensors.M_K)


Mobility Matrix M_H (multiplied by mu):
[[ 0.   ]
 [ 0.   ]
 [ 0.   ]
 [ 0.   ]
 [ 0.   ]
 [ 0.002]
 [ 0.037]]

Mobility Matrix M_K (multiplied by mu):
[[ 0.   ]
 [-0.141]
 [ 0.   ]
 [ 0.   ]
 [ 0.   ]
 [ 0.168]
 [-0.542]]


In [104]:
m1 = tensors.M_H[1,0]
m2 = tensors.M_H[-2,0]
m3 = tensors.M_H[-1,0]

k1 = tensors.M_K[1,0] 
k2 = tensors.M_K[-2,0]
k3 = tensors.M_K[-1,0]

print("Magnetic coupling coefficients:", m1, m2, m3)
print("Elastic  coupling coefficients:", k1, k2, k3) 

Magnetic coupling coefficients: 9.2426315e-05 0.0023460574 0.03739889
Elastic  coupling coefficients: -0.14099357 0.1676875 -0.5416764


In the limit of small angles we have
$$ \dot \theta_0 = -m_2 F (\theta_0 + \phi) + m_2 F R \sin(\omega t) + k_2 \phi, $$
$$ \dot \phi     = -m_3 F (\theta_0 + \phi) + m_3 F R \sin(\omega t) + k_3 \phi, $$
with $F = m B_x /(\mu a \omega)$, the amplitude of the magnetic forcing (along the steady direction), and $R = B_y/ B_x$ the ratio of magnetic components between the oscillating and steady ones. 

Calculating mean velocity in this limit with Wolfram calculation `Magnetic soft robot oneDOF`.

In [105]:
vmoy = (MAGNETIC_FORCING**2*(k2*m1 - k1*m2)*m3)/(2*((1 + k3**2)*(1 + MAGNETIC_FORCING**2*m2**2) - 2*MAGNETIC_FORCING*(k2 + k3 + MAGNETIC_FORCING*(-1 + k2*k3)*m2)*m3 + MAGNETIC_FORCING**2*(1 + k2**2)*m3**2))

print("Mean velocity normalized by R**2:", vmoy)
print("Distance for 5 periods:", vmoy * 10 * np.pi * 0.1**2)

Mean velocity normalized by R**2: 0.0031653966
Distance for 5 periods: 0.0009944388


### Optimization

In [106]:
def optimize_velocity(
    init_design,
    learning_rate=0.03,
    max_iters=1000,
    momentum = 0.9,  # Tune this (0.8 - 0.95 works well)
    forcing = MAGNETIC_FORCING
):
    def mean_v(design):
        dofs = jnp.zeros(theory_body.Ndof)
        tensors = theory_body.compute_mobility_problem(dofs, design)

        m1 = tensors.M_H[1,0]
        m2 = tensors.M_H[-2,0]
        m3 = tensors.M_H[-1,0]

        k1 = tensors.M_K[1,0]
        k2 = tensors.M_K[-2,0]
        k3 = tensors.M_K[-1,0]
        
        vmean = (forcing**2*(k2*m1 - k1*m2)*m3)/(2*((1 + k3**2)*(1 + forcing**2*m2**2) - 2*forcing*(k2 + k3 + forcing*(-1 + k2*k3)*m2)*m3 + forcing**2*(1 + k2**2)*m3**2))

        return vmean

    def constraints_design(design):
        return jnp.clip(design, 0, 1)

    design = init_design
    grad_mean_v = jax.jit(jax.grad(mean_v))
    velocity = jnp.zeros_like(design) 
        
    for i in range(max_iters):
        grad = grad_mean_v(design)
        grad_norm = jnp.linalg.norm(grad)

        if i % 100 == 0:
            print(i, mean_v(design), design, grad_norm)

        # Update velocity (exponential moving average of gradients)
        velocity = momentum * velocity + learning_rate * grad / (grad_norm + 1E-6)
        design = design + velocity

        # Enforce any additional constraints (like bounding design between 0 and 1)
        design = constraints_design(design)

    return design, mean_v(design)    

In [107]:
# Optimization trial
init_params = theory_body.design_defaults
opt_design, value = optimize_velocity(
    init_design=init_params,
    learning_rate=1E-3, 
    max_iters=2001,
    forcing=100,
)

0 0.0031653966 [ 0.5   0.25  0.1 ] 0.02730152
100 0.01596637 [ 0.     0.328  0.353] 0.024827765
200 0.015993915 [ 0.     0.334  0.381] 0.025289837
300 0.015993943 [ 0.     0.334  0.382] 0.025320549
400 0.015993943 [ 0.     0.334  0.382] 0.025321886
500 0.015993945 [ 0.     0.334  0.382] 0.025321912
600 0.015993945 [ 0.     0.334  0.382] 0.025321912
700 0.015993945 [ 0.     0.334  0.382] 0.025321912
800 0.015993945 [ 0.     0.334  0.382] 0.025321912
900 0.015993945 [ 0.     0.334  0.382] 0.025321912
1000 0.015993945 [ 0.     0.334  0.382] 0.025321912
1100 0.015993945 [ 0.     0.334  0.382] 0.025321912
1200 0.015993945 [ 0.     0.334  0.382] 0.025321912
1300 0.015993945 [ 0.     0.334  0.382] 0.025321912
1400 0.015993945 [ 0.     0.334  0.382] 0.025321912
1500 0.015993945 [ 0.     0.334  0.382] 0.025321912
1600 0.015993945 [ 0.     0.334  0.382] 0.025321912
1700 0.015993945 [ 0.     0.334  0.382] 0.025321912
1800 0.015993945 [ 0.     0.334  0.382] 0.025321912
1900 0.015993945 [ 0.     0.

In [108]:
print(theory_body.design_variables)
print(opt_design * np.array([2, 2, 100]))

['length', 'radius', 'stiffness']
[  0.      0.669  38.247]


### Optimization for different magnetic forcing amplitudes

In [45]:
forcings = 10**np.linspace(start=4, stop=-1, num=20,endpoint=True)
opt_length = np.zeros_like(forcings)
opt_radius = np.zeros_like(forcings)
opt_stiffness = np.zeros_like(forcings)
opt_speed = np.zeros_like(forcings)

init_design = opt_design
for i, f in enumerate(forcings):
    init_design, value = optimize_velocity(
        init_design=init_design,
        learning_rate=1E-3, 
        max_iters=201,
        forcing=f,
    )
    print(i, f, init_design)
    opt_length[i] = init_design[0] * 2
    opt_radius[i] = init_design[1] * 2
    opt_stiffness[i] = init_design[2] * 100
    opt_speed[i] = value

0 0.026714485 [ 0.     0.511  0.83 ] 0.04193185
100 0.026714891 [ 0.     0.511  0.833] 0.041929845
200 0.026715202 [ 0.     0.512  0.835] 0.04192781
0 10000.0 [ 0.     0.512  0.835]
0 0.026556903 [ 0.     0.512  0.835] 0.04192189
100 0.026557375 [ 0.     0.51   0.834] 0.04174716
200 0.02655739 [ 0.     0.51   0.833] 0.041749023
1 5455.59478116852 [ 0.     0.51   0.833]
0 0.02626956 [ 0.     0.51   0.833] 0.041741285
100 0.026272522 [ 0.     0.506  0.828] 0.041424785
200 0.026273599 [ 0.     0.505  0.823] 0.0414308
2 2976.351441631319 [ 0.     0.505  0.823]
0 0.025757201 [ 0.     0.505  0.823] 0.041426275
100 0.02576887 [ 0.     0.498  0.811] 0.0408508
200 0.025773387 [ 0.     0.496  0.801] 0.04086042
3 1623.776739188721 [ 0.     0.496  0.801]
0 0.024867056 [ 0.     0.496  0.801] 0.040875528
100 0.024903938 [ 0.     0.483  0.779] 0.03983635
200 0.024918053 [ 0.     0.479  0.761] 0.03984165
4 885.8667904100832 [ 0.     0.479  0.76 ]
0 0.023385067 [ 0.     0.479  0.76 ] 0.039906904
100 0.

In [46]:
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
)

# Subplot 1: length and radius
fig.add_trace(go.Scatter(x=forcings, y=opt_length,    mode='lines+markers', name='Length'), row=1, col=1)
fig.add_trace(go.Scatter(x=forcings, y=opt_radius,    mode='lines+markers', name='Radius'), row=1, col=1)

# Subplot 2: stiffness
fig.add_trace(go.Scatter(x=forcings, y=opt_stiffness, mode='lines+markers', name='Stiffness'), row=2, col=1)

# Subplot 3: speed
fig.add_trace(go.Scatter(x=forcings, y=opt_speed, mode='lines+markers', name='Speed'), row=3, col=1)


axis_style = dict(
    showline=True, linewidth=1, linecolor="black", mirror=True,
    showgrid=False, zeroline=False,
    ticks="inside", tickwidth=1, tickcolor="black",
)

fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="serif", size=12, color="black"),
    height=1000, width=600,
    legend=dict(bordercolor="black", borderwidth=1, bgcolor="white"),
    xaxis= dict(**axis_style, showticklabels=False, type="log"),
    xaxis2= dict(**axis_style, showticklabels=False, type="log"),
    xaxis3=dict(**axis_style, title="Magnetic Forcing",      type="log"),
    yaxis= dict(**axis_style, title="Length, Radius"),
    yaxis2=dict(**axis_style, title="Stiffness"),
    yaxis3=dict(**axis_style, title="Speed", type="log")
)

fig.show()

### Limit of large magnetic fields

In [52]:
def optimize_velocity_largeB(
    init_design,
    learning_rate=0.03,
    max_iters=1000,
    momentum = 0.9,  # Tune this (0.8 - 0.95 works well)
):
    def mean_v(design):
        dofs = jnp.zeros(theory_body.Ndof)
        tensors = theory_body.compute_mobility_problem(dofs, design)

        m1 = tensors.M_H[1,0]
        m2 = tensors.M_H[-2,0]
        m3 = tensors.M_H[-1,0]

        k1 = tensors.M_K[1,0]
        k2 = tensors.M_K[-2,0]
        k3 = tensors.M_K[-1,0]
        
        vmean = ((k2*m1 - k1*m2)*m3)/(2*((1 + k3**2)*m2**2 - 2*(-1 + k2*k3)*m2*m3 + (1 + k2**2)*m3**2))

        return vmean

    def constraints_design(design):
        return jnp.clip(design, 0, 1)

    design = init_design
    grad_mean_v = jax.jit(jax.grad(mean_v))
    velocity = jnp.zeros_like(design) 
        
    for i in range(max_iters):
        grad = grad_mean_v(design)
        grad_norm = jnp.linalg.norm(grad)

        if i % 100 == 0:
            print(i, mean_v(design), design, grad_norm)

        # Update velocity (exponential moving average of gradients)
        velocity = momentum * velocity + learning_rate * grad / (grad_norm + 1E-6)
        design = design + velocity

        # Enforce any additional constraints (like bounding design between 0 and 1)
        design = constraints_design(design)

    return design, mean_v(design)    

In [53]:
# Optimization trial
init_params = theory_body.design_defaults
opt_design, value = optimize_velocity_largeB(
    init_design=init_params,
    learning_rate=1E-3, 
    max_iters=2001,
)

0 0.0039566304 [ 0.5   0.25  0.1 ] 0.03949316
100 0.025630837 [ 0.     0.427  0.506] 0.041510284
200 0.026571408 [ 0.     0.465  0.653] 0.041504838
300 0.026773171 [ 0.     0.484  0.721] 0.041946936
400 0.026846182 [ 0.     0.495  0.762] 0.042086594
500 0.026877826 [ 0.     0.502  0.789] 0.042134497
600 0.026892822 [ 0.     0.507  0.808] 0.04214918
700 0.026900424 [ 0.     0.51   0.821] 0.042151164
800 0.026904343 [ 0.     0.512  0.83 ] 0.042148564
900 0.026906442 [ 0.     0.514  0.837] 0.042144474
1000 0.026907591 [ 0.     0.515  0.842] 0.042140413
1100 0.02690823 [ 0.     0.516  0.846] 0.042136773
1200 0.026908565 [ 0.     0.517  0.849] 0.04213376
1300 0.026908766 [ 0.     0.518  0.851] 0.04213131
1400 0.026908867 [ 0.     0.518  0.853] 0.04212931
1500 0.026908945 [ 0.     0.518  0.854] 0.0421279
1600 0.026908997 [ 0.     0.519  0.855] 0.042126674
1700 0.026908996 [ 0.     0.519  0.855] 0.042125754
1800 0.026909016 [ 0.     0.519  0.856] 0.04212512
1900 0.02690904 [ 0.     0.519  0.8

In [ ]:
print(theory_body.design_variables)
print(opt_design * np.array([2, 2, 100]))

['length', 'radius', 'stiffness']
[  0.      1.027  84.185]
